In [ ]:
# ======================================
# PART 1: Import Library
# V6: Added correlation_fixes_v6 module

from scapy.all import IP, IPv6, TCP, Ether, Padding, wrpcap, raw, rdpcap, load_contrib
from scapy.contrib.bgp import *
from scapy.utils import PcapReader
from scipy.stats import pareto, weibull_min
import datetime
import time
import random
import os
import csv
import struct
import traceback
from collections import defaultdict
from typing import Dict, List, Tuple, Set, Optional

# V6: Import correlation fixes
from correlation_fixes_v6 import (
    PrefixBehaviorProfileV6,
    sample_prefix_behavior_profile_v6,
    PrefixStateTrackerV6,
    generate_traffic_v6,
    sample_edit_distance_v6,
    vary_as_path_v6,
    calculate_edit_distance,
    generate_as_path_v6,
    generate_withdrawal_nadas_sequence_v6,
    generate_flapping_sequence_v6,
    generate_imp_wd_spath_withdrawal_sequence_v6,
    generate_imp_wd_withdrawal_sequence_v6,
    generate_standalone_duplicates_v6,
    generate_edit_distance_cluster_sequence_v6,
    print_v6_summary,
    REAL_CORRELATIONS,
)

load_contrib('bgp')

# Create output directory for pcap files
os.makedirs('./pcaps', exist_ok=True)
print('✅ Output directory created: pcaps')

# Print V6 summary
print_v6_summary()

In [ ]:
# ======================================
# PART 2: AS Topology Generation (ENHANCED)

def generate_as_topology():
    """Generate a realistic BGP topology with hierarchical AS structure using specific AS numbers"""
    
    # Use SPECIFIC AS numbers for a more realistic Internet topology
    as_numbers = {
        'tier1': [1299, 3356, 174, 3257, 6762],  # Major Tier-1 ISPs
        'tier2': [6939, 1273, 3320, 6453, 2914, 5511, 7018],  # Large Regional ISPs
        'tier3': [
            41336, 35060,  # Your target ASes for attack scenarios
            34554, 49544, 50673, 39126, 48292, 62041,
            45899, 51697, 60781, 44002, 56630, 31027, 64512
        ],  # Edge networks and small ISPs
        'ixp_content': [13335, 15169, 32934]  # Cloudflare, Google, Facebook
    }
    
    # Initialize topology structure
    topology = {}
    
    # Create basic structure for all ASes
    all_asns = []
    for tier, asn_list in as_numbers.items():
        tier_level = int(tier.replace('tier', '')) if 'tier' in tier else 4
        for asn in asn_list:
            all_asns.append(asn)
            topology[asn] = {
                'tier': tier_level,
                'neighbors': [],
                'relationships': {}
            }
    
    # Connect Tier 1s to each other (full mesh)
    for i, asn1 in enumerate(as_numbers['tier1']):
        for asn2 in as_numbers['tier1'][i+1:]:
            topology[asn1]['neighbors'].append(asn2)
            topology[asn2]['neighbors'].append(asn1)
            topology[asn1]['relationships'][asn2] = 'peer'
            topology[asn2]['relationships'][asn1] = 'peer'
    
    # Connect Tier 2s to Tier 1s
    for asn2 in as_numbers['tier2']:
        num_providers = random.randint(1, min(3, len(as_numbers['tier1'])))
        providers = random.sample(as_numbers['tier1'], num_providers)
        for asn1 in providers:
            topology[asn2]['neighbors'].append(asn1)
            topology[asn1]['neighbors'].append(asn2)
            topology[asn2]['relationships'][asn1] = 'provider'
            topology[asn1]['relationships'][asn2] = 'customer'
    
    # Connect some Tier 2s to each other
    for i, asn1 in enumerate(as_numbers['tier2']):
        num_peers = random.randint(1, 3)
        potential_peers = as_numbers['tier2'][i+1:]
        if potential_peers:
            peers = random.sample(potential_peers, min(num_peers, len(potential_peers)))
            for asn2 in peers:
                if asn2 not in topology[asn1]['neighbors']:
                    topology[asn1]['neighbors'].append(asn2)
                    topology[asn2]['neighbors'].append(asn1)
                    topology[asn1]['relationships'][asn2] = 'peer'
                    topology[asn2]['relationships'][asn1] = 'peer'
    
    # Connect Tier 3s to Tier 2s
    for asn3 in as_numbers['tier3']:
        num_providers = random.randint(1, min(2, len(as_numbers['tier2'])))
        providers = random.sample(as_numbers['tier2'], num_providers)
        for asn2 in providers:
            topology[asn3]['neighbors'].append(asn2)
            topology[asn2]['neighbors'].append(asn3)
            topology[asn3]['relationships'][asn2] = 'provider'
            topology[asn2]['relationships'][asn3] = 'customer'
    
    # Connect IXPs to multiple ASes
    for ixp_asn in as_numbers['ixp_content']:
        tier2_connections = random.sample(as_numbers['tier2'], min(4, len(as_numbers['tier2'])))
        tier3_connections = random.sample(as_numbers['tier3'], min(3, len(as_numbers['tier3'])))
        for asn in tier2_connections + tier3_connections:
            topology[ixp_asn]['neighbors'].append(asn)
            topology[asn]['neighbors'].append(ixp_asn)
            topology[ixp_asn]['relationships'][asn] = 'peer'
            topology[asn]['relationships'][ixp_asn] = 'peer'
    
    main_src_as = 41336
    main_dst_as = 35060
    
    if main_dst_as not in topology[main_src_as]['neighbors']:
        topology[main_src_as]['neighbors'].append(main_dst_as)
        topology[main_dst_as]['neighbors'].append(main_src_as)
        topology[main_src_as]['relationships'][main_dst_as] = 'peer'
        topology[main_dst_as]['relationships'][main_src_as] = 'peer'
    
    return topology, as_numbers, main_src_as, main_dst_as

topology, as_numbers, main_src_as, main_dst_as = generate_as_topology()

print(f'Generated AS topology with {len(topology)} ASes')
print(f'Main AS pair: AS{main_src_as} and AS{main_dst_as}')

In [ ]:
# ======================================
# PART 3: IP and Interface Allocation

# Special prefixes for scenarios
PREDEFINED_PREFIXES = [
    '203.0.113.0/24',
    '198.51.100.0/24',
    '192.0.2.0/24'
]

# V6: Extended prefix pool for better diversity
EXTENDED_PREFIXES = PREDEFINED_PREFIXES + [
    f'10.{i}.{j}.0/24' for i in range(10) for j in range(10)
] + [
    f'172.16.{i}.0/24' for i in range(50)
] + [
    f'192.168.{i}.0/24' for i in range(50)
]

def allocate_ip_addresses(topology, as_numbers, main_src_as, main_dst_as):
    ip_allocations = {}
    
    for tier, asn_list in as_numbers.items():
        for asn in asn_list:
            if tier == 'tier1':
                octet1, octet2 = 100, random.randint(64, 127)
            elif tier == 'tier2':
                octet1, octet2 = 172, random.randint(16, 31)
            elif tier == 'tier3':
                octet1, octet2 = 192, 168
            else:
                octet1, octet2 = 10, random.randint(0, 255)
            
            octet3 = random.randint(0, 255)
            router_id = f'{octet1}.{octet2}.{octet3}.1'
            
            announced_prefixes = []
            if asn == main_src_as:
                announced_prefixes = PREDEFINED_PREFIXES.copy()
            else:
                if tier == 'tier1':
                    for _ in range(random.randint(1, 2)):
                        prefix = f'203.{random.randint(0, 254)}.0.0/16'
                        if prefix != '203.0.113.0/16':
                            announced_prefixes.append(prefix)
                elif tier == 'tier2':
                    for _ in range(random.randint(1, 3)):
                        prefix = f'198.51.{random.randint(0, 99)}.0/24'
                        if prefix != '198.51.100.0/24':
                            announced_prefixes.append(prefix)
                elif tier == 'tier3':
                    for _ in range(random.randint(1, 2)):
                        third = random.randint(3, 255)
                        prefix = f'192.0.{third}.0/24'
                        if prefix != '192.0.2.0/24':
                            announced_prefixes.append(prefix)
                else:
                    prefix = f'198.18.{random.randint(0, 255)}.0/24'
                    announced_prefixes.append(prefix)
                
                if not announced_prefixes:
                    announced_prefixes.append(f'172.{random.randint(20, 30)}.{random.randint(0, 255)}.0/24')
            
            ip_allocations[asn] = {
                'router_id': router_id,
                'announced_prefixes': announced_prefixes,
                'interfaces': {}
            }
    
    for asn, info in topology.items():
        for neighbor in info['neighbors']:
            if neighbor in ip_allocations[asn]['interfaces']:
                continue
            link_net1 = random.randint(0, 255)
            link_net2 = random.randint(0, 255)
            link_net3 = random.randint(0, 63) * 4
            if asn < neighbor:
                ip_allocations[asn]['interfaces'][neighbor] = f'10.{link_net1}.{link_net2}.{link_net3+1}'
                ip_allocations[neighbor]['interfaces'][asn] = f'10.{link_net1}.{link_net2}.{link_net3+2}'
            else:
                ip_allocations[asn]['interfaces'][neighbor] = f'10.{link_net1}.{link_net2}.{link_net3+2}'
                ip_allocations[neighbor]['interfaces'][asn] = f'10.{link_net1}.{link_net2}.{link_net3+1}'
    
    return ip_allocations

ip_allocations = allocate_ip_addresses(topology, as_numbers, main_src_as, main_dst_as)

NORMAL_TRAFFIC_ID_RANGE = (0x03E8, 0x7527)
PREFIX_HIJACK_ID_RANGE = (0x7530, 0x9C3F)
PATH_MANIP_ID_RANGE = (0x9C40, 0xC34F)
DOS_ATTACK_ID_RANGE = (0xC350, 0xEA5F)
ROUTE_LEAK_ID_RANGE = (0xEA60, 0xFFFF)

print(f'IP allocations complete for {len(ip_allocations)} ASes')

In [ ]:
# ======================================
# PART 4: V6 State Tracker for Packet Generation
# This bridges between correlation_fixes_v6 and Scapy packet generation

class BGPStateTrackerV6:
    """
    V6 State tracker that bridges correlation_fixes_v6 with Scapy.
    """
    
    def __init__(self):
        self.prefix_state = PrefixStateTrackerV6()
        self.peer_states = defaultdict(lambda: {
            'announced_prefixes': {},
            'last_as_path': {},
        })
    
    def announce_prefix(self, peer_ip, prefix, as_path, timestamp=0.0):
        """Track announcement and return event type"""
        event_type, edit_dist = self.prefix_state.announce(prefix, as_path)
        self.peer_states[peer_ip]['announced_prefixes'][prefix] = True
        self.peer_states[peer_ip]['last_as_path'][prefix] = as_path
        return event_type, edit_dist
    
    def withdraw_prefix(self, peer_ip, prefix, timestamp=0.0):
        """Track withdrawal"""
        was_announced = self.prefix_state.withdraw(prefix)
        if prefix in self.peer_states[peer_ip]['announced_prefixes']:
            del self.peer_states[peer_ip]['announced_prefixes'][prefix]
        return was_announced
    
    def get_profile(self, prefix):
        return self.prefix_state.get_or_create_profile(prefix)

# Initialize global tracker
global_state_tracker = BGPStateTrackerV6()

print('✅ BGPStateTrackerV6 initialized')

In [ ]:
# ======================================
# PART 5: BGP Session and Packet Generation

pkts = []

def create_mac_from_ip(ip):
    octets = list(map(int, ip.split('.')))
    return f'00:{octets[0]:02x}:{octets[1]:02x}:{octets[2]:02x}:{octets[3]:02x}:00'

def create_bgp_packet_base(src_mac, dst_mac, src_ip, dst_ip, src_port, dst_port, seq, ack, ip_id, ttl=1):
    return Ether(src=src_mac, dst=dst_mac)/IP(src=src_ip, dst=dst_ip, ttl=ttl, flags='DF', tos=0xC0, id=ip_id)/TCP(sport=src_port, dport=dst_port, flags='PA', seq=seq, ack=ack, window=16384)

def pad_packet(pkt, min_size=60):
    if len(pkt) < min_size:
        pad_len = min_size - len(pkt)
        pkt = pkt/Padding(load=b'\\x00' * pad_len)
    return pkt

def create_as_path_attribute(as_path):
    """Create BGP AS_PATH attribute from list of ASNs"""
    as_path_attr = BGPPathAttr(type_flags=0x40, type_code=2)
    as_path_segment = BGPPAASPath()
    segment = BGPPAASPath.ASPathSegment(
        segment_type=2,
        segment_length=len(as_path),
        segment_value=as_path
    )
    as_path_segment.segments = [segment]
    as_path_attr.attribute = as_path_segment
    return as_path_attr

def create_standard_attributes(src_asn, src_ip, as_path, med=100, local_pref=200, origin=0):
    """Create standard BGP path attributes"""
    origin_attr = BGPPathAttr(type_flags=0x40, type_code=1)
    origin_attr.attribute = BGPPAOrigin(origin=origin)
    
    as_path_attr = create_as_path_attribute(as_path)
    
    next_hop_attr = BGPPathAttr(type_flags=0x40, type_code=3)
    next_hop_attr.attribute = BGPPANextHop(next_hop=src_ip)
    
    med_attr = BGPPathAttr(type_flags=0x80, type_code=4)
    med_attr.attribute = BGPPAMultiExitDisc(med=med)
    
    local_pref_attr = BGPPathAttr(type_flags=0x40, type_code=5)
    local_pref_attr.attribute = BGPPALocalPref(local_pref=local_pref)
    
    communities_list = [BGPPACommunity(community=0xFFFFFF01), BGPPACommunity(community=src_asn<<16|200)]
    communities_attr = BGPPathAttr(type_flags=0x40|0x80, type_code=8)
    communities_attr.attribute = communities_list
    
    return [origin_attr, as_path_attr, next_hop_attr, med_attr, local_pref_attr, communities_attr]

def generate_all_bgp_sessions(topology, ip_allocations):
    bgp_sessions_ipv4 = {}
    bgp_sessions_ipv6 = {}
    all_packets = []
    
    for asn, info in topology.items():
        for neighbor in info['neighbors']:
            if (asn, neighbor) in bgp_sessions_ipv4:
                continue
            
            src_ipv4 = ip_allocations[asn]['interfaces'][neighbor]
            dst_ipv4 = ip_allocations[neighbor]['interfaces'][asn]
            src_router_id = ip_allocations[asn]['router_id']
            dst_router_id = ip_allocations[neighbor]['router_id']
            
            src_mac = '00:' + ':'.join([f'{random.randint(0, 255):02x}' for _ in range(5)])
            dst_mac = '00:' + ':'.join([f'{random.randint(0, 255):02x}' for _ in range(5)])
            
            src_port = random.randint(30000, 65000)
            dst_port = 179
            
            seq_a_v4 = random.randint(1000, 10000)
            seq_b_v4 = random.randint(1000, 10000)
            
            src_ip_id = random.randint(NORMAL_TRAFFIC_ID_RANGE[0], NORMAL_TRAFFIC_ID_RANGE[1])
            dst_ip_id = random.randint(NORMAL_TRAFFIC_ID_RANGE[0], NORMAL_TRAFFIC_ID_RANGE[1])
            
            # TCP 3-way handshake
            tcp_options = [('MSS', 1460)]
            
            syn_pkt = Ether(src=src_mac, dst=dst_mac)/IP(src=src_ipv4, dst=dst_ipv4, ttl=1, flags='DF', tos=0xC0, id=src_ip_id)/TCP(sport=src_port, dport=dst_port, flags='S', seq=seq_a_v4, window=16384, options=tcp_options)
            syn_pkt = pad_packet(syn_pkt)
            all_packets.append(syn_pkt)
            
            synack_pkt = Ether(src=dst_mac, dst=src_mac)/IP(src=dst_ipv4, dst=src_ipv4, ttl=1, flags=0, tos=0xC0, id=dst_ip_id)/TCP(sport=dst_port, dport=src_port, flags='SA', seq=seq_b_v4, ack=seq_a_v4+1, window=16384, options=tcp_options)
            synack_pkt = pad_packet(synack_pkt)
            all_packets.append(synack_pkt)
            
            ack_pkt = Ether(src=src_mac, dst=dst_mac)/IP(src=src_ipv4, dst=dst_ipv4, ttl=1, flags='DF', tos=0xC0, id=src_ip_id+1)/TCP(sport=src_port, dport=dst_port, flags='A', seq=seq_a_v4+1, ack=seq_b_v4+1, window=16384)
            ack_pkt = pad_packet(ack_pkt)
            all_packets.append(ack_pkt)
            
            seq_a_v4 += 1
            seq_b_v4 += 1
            
            # BGP OPEN
            mp_ipv4_cap = BGPCapMultiprotocol(code=1, length=4, afi=1, safi=1)
            mp_ipv6_cap = BGPCapMultiprotocol(code=1, length=4, afi=2, safi=1)
            as4_cap = BGPCapFourBytesASN(code=65, length=4, asn=asn)
            
            opt_params = [
                BGPOptParam(param_type=2, param_length=len(mp_ipv4_cap), param_value=mp_ipv4_cap),
                BGPOptParam(param_type=2, param_length=len(mp_ipv6_cap), param_value=mp_ipv6_cap),
                BGPOptParam(param_type=2, param_length=len(as4_cap), param_value=as4_cap)
            ]
            
            open_msg = BGPHeader(type=1)/BGPOpen(version=4, my_as=asn, hold_time=180, bgp_id=src_router_id, opt_params=opt_params)
            open_pkt = Ether(src=src_mac, dst=dst_mac)/IP(src=src_ipv4, dst=dst_ipv4, ttl=1, flags='DF', tos=0xC0, id=src_ip_id+2)/TCP(sport=src_port, dport=dst_port, flags='PA', seq=seq_a_v4, ack=seq_b_v4, window=16384)/open_msg
            open_pkt = pad_packet(open_pkt)
            all_packets.append(open_pkt)
            seq_a_v4 += len(open_msg)
            
            # KEEPALIVE
            keepalive = BGPKeepAlive()
            keepalive_pkt = Ether(src=src_mac, dst=dst_mac)/IP(src=src_ipv4, dst=dst_ipv4, ttl=1, flags='DF', tos=0xC0, id=src_ip_id+3)/TCP(sport=src_port, dport=dst_port, flags='PA', seq=seq_a_v4, ack=seq_b_v4, window=16384)/keepalive
            keepalive_pkt = pad_packet(keepalive_pkt)
            all_packets.append(keepalive_pkt)
            seq_a_v4 += len(keepalive)
            
            bgp_sessions_ipv4[(asn, neighbor)] = {
                'src_ipv4': src_ipv4,
                'dst_ipv4': dst_ipv4,
                'src_mac': src_mac,
                'dst_mac': dst_mac,
                'seq_a': seq_a_v4,
                'seq_b': seq_b_v4,
                'sport': src_port,
                'dport': dst_port,
                'src_ip_id': src_ip_id + 4,
                'dst_ip_id': dst_ip_id + 1,
                'packets': []
            }
    
    print(f'Generated {len(all_packets)} session establishment packets')
    return bgp_sessions_ipv4, bgp_sessions_ipv6, all_packets

global_bgp_sessions_ipv4, global_bgp_sessions_ipv6, session_pkts = generate_all_bgp_sessions(topology, ip_allocations)
pkts.extend(session_pkts)

print(f'Created {len(global_bgp_sessions_ipv4)} BGP sessions')

In [ ]:
# ======================================
# PART 6: V6 Traffic Generation with Correlation Fixes
# This is the KEY CELL that generates traffic with proper correlations

print('[+] Generating V6 traffic with correlation fixes...')

# Get AS pools
tier1_ases = as_numbers['tier1']
tier2_ases = as_numbers['tier2']
tier3_ases = as_numbers['tier3']

# Create rare AS pool (for NADAS)
rare_as_pool = list(range(64512, 65534)) + list(range(400000, 410000))

# Get main session info
session_key = (main_src_as, main_dst_as)
if session_key not in global_bgp_sessions_ipv4:
    session_key = (main_dst_as, main_src_as)

session = global_bgp_sessions_ipv4[session_key]
src_ipv4 = session['src_ipv4']
dst_ipv4 = session['dst_ipv4']
src_mac = session['src_mac']
dst_mac = session['dst_mac']
sport = session['sport']
dport = session['dport']
seq_a = session['seq_a']
seq_b = session['seq_b']
src_ip_id = session['src_ip_id']
dst_ip_id = session['dst_ip_id']

# Generate V6 traffic events
TARGET_EVENTS = 500  # Adjust this for more/less traffic

events, tracker = generate_traffic_v6(
    peer_ip=src_ipv4,
    tier1_ases=tier1_ases,
    tier2_ases=tier2_ases,
    rare_as_pool=rare_as_pool,
    predefined_prefixes=EXTENDED_PREFIXES,
    target_events=TARGET_EVENTS
)

print(f'Generated {len(events)} events from V6 traffic generator')

# Convert events to packets
v6_packets = []
event_stats = defaultdict(int)

for event in events:
    if event['action'] == 'announce':
        as_path = event.get('as_path', [main_src_as])
        prefix = event['prefix']
        
        # Create UPDATE packet
        update = BGPHeader(type=2)/BGPUpdate()
        
        # Determine origin based on event
        origin_val = 0  # IGP by default
        if event.get('is_nadas') or event.get('is_flap'):
            origin_val = random.choice([0, 2])  # Sometimes INCOMPLETE for NADAS/flaps
        
        path_attrs = create_standard_attributes(main_src_as, src_ipv4, as_path, origin=origin_val)
        update.path_attr = path_attrs
        update.nlri.append(BGPNLRI_IPv4(prefix=prefix))
        
        pkt = Ether(src=src_mac, dst=dst_mac)/IP(src=src_ipv4, dst=dst_ipv4, ttl=1, flags='DF', tos=0xC0, id=src_ip_id)/TCP(sport=sport, dport=dport, flags='PA', seq=seq_a, ack=seq_b, window=16384)/update
        pkt = pad_packet(pkt)
        v6_packets.append(pkt)
        
        seq_a += len(update)
        src_ip_id += 1
        
        # ACK
        ack_pkt = Ether(src=dst_mac, dst=src_mac)/IP(src=dst_ipv4, dst=src_ipv4, ttl=1, flags=0, tos=0xC0, id=dst_ip_id)/TCP(sport=dport, dport=sport, flags='A', seq=seq_b, ack=seq_a, window=16384)
        ack_pkt = pad_packet(ack_pkt)
        v6_packets.append(ack_pkt)
        dst_ip_id += 1
        
        event_stats[event.get('event_type', 'announce')] += 1
        
    elif event['action'] == 'withdraw':
        prefix = event['prefix']
        
        # Create WITHDRAW UPDATE
        withdraw = BGPHeader(type=2)/BGPUpdate()
        withdraw.withdrawn_routes.append(BGPNLRI_IPv4(prefix=prefix))
        
        pkt = Ether(src=src_mac, dst=dst_mac)/IP(src=src_ipv4, dst=dst_ipv4, ttl=1, flags='DF', tos=0xC0, id=src_ip_id)/TCP(sport=sport, dport=dport, flags='PA', seq=seq_a, ack=seq_b, window=16384)/withdraw
        pkt = pad_packet(pkt)
        v6_packets.append(pkt)
        
        seq_a += len(withdraw)
        src_ip_id += 1
        
        # ACK
        ack_pkt = Ether(src=dst_mac, dst=src_mac)/IP(src=dst_ipv4, dst=src_ipv4, ttl=1, flags=0, tos=0xC0, id=dst_ip_id)/TCP(sport=dport, dport=sport, flags='A', seq=seq_b, ack=seq_a, window=16384)
        ack_pkt = pad_packet(ack_pkt)
        v6_packets.append(ack_pkt)
        dst_ip_id += 1
        
        event_stats['withdrawal'] += 1

pkts.extend(v6_packets)

print(f'\n✅ Generated {len(v6_packets)} V6 packets')
print(f'\nEvent Statistics:')
for event_type, count in sorted(event_stats.items()):
    print(f'  {event_type}: {count}')

In [ ]:
# ======================================
# PART 7: Save PCAP and Generate CSV

OUTPUT_PCAP = './pcaps/bgp_v6_correlated.pcap'
OUTPUT_CSV = './pcaps/bgp_v6_correlated.csv'

print(f'[+] Writing {len(pkts)} packets to {OUTPUT_PCAP}...')
wrpcap(OUTPUT_PCAP, pkts)
print(f'✅ PCAP saved to {OUTPUT_PCAP}')

# Extract to CSV
def extract_bgp_to_csv(pcap_file, csv_file):
    load_contrib('bgp')
    
    rows = []
    packets = rdpcap(pcap_file)
    
    for pkt in packets:
        if not pkt.haslayer(TCP) or pkt[TCP].dport != 179 and pkt[TCP].sport != 179:
            continue
        
        if not pkt.haslayer(BGPHeader):
            continue
        
        bgp = pkt.getlayer(BGPHeader)
        
        if bgp.type != 2:  # UPDATE
            continue
        
        update = bgp.getlayer(BGPUpdate)
        if not update:
            continue
        
        timestamp = float(pkt.time)
        time_str = datetime.datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S.%f')
        
        if pkt.haslayer(IP):
            peer_ip = pkt[IP].src
        else:
            peer_ip = pkt[IPv6].src
        
        # Extract path attributes
        as_path = ''
        origin = ''
        next_hop = ''
        med = ''
        local_pref = ''
        communities = ''
        peer_asn = ''
        
        for attr in update.path_attr:
            if attr.type_code == 1:  # ORIGIN
                origin_val = attr.attribute.origin if hasattr(attr.attribute, 'origin') else 0
                origin = ['IGP', 'EGP', 'INCOMPLETE'][origin_val] if origin_val < 3 else str(origin_val)
            elif attr.type_code == 2:  # AS_PATH
                if hasattr(attr.attribute, 'segments'):
                    path_asns = []
                    for seg in attr.attribute.segments:
                        path_asns.extend(seg.segment_value)
                    as_path = ' '.join(map(str, path_asns))
                    if path_asns:
                        peer_asn = path_asns[-1]
            elif attr.type_code == 3:  # NEXT_HOP
                next_hop = attr.attribute.next_hop if hasattr(attr.attribute, 'next_hop') else ''
            elif attr.type_code == 4:  # MED
                med = attr.attribute.med if hasattr(attr.attribute, 'med') else ''
            elif attr.type_code == 5:  # LOCAL_PREF
                local_pref = attr.attribute.local_pref if hasattr(attr.attribute, 'local_pref') else ''
            elif attr.type_code == 8:  # COMMUNITIES
                comms = []
                for c in attr.attribute:
                    if hasattr(c, 'community'):
                        comms.append(str(c.community))
                communities = ' '.join(comms)
        
        # Announcements
        for nlri in update.nlri:
            prefix = nlri.prefix if hasattr(nlri, 'prefix') else str(nlri)
            rows.append({
                'Time': time_str,
                'Entry_Type': 'A',
                'Peer_IP': peer_ip,
                'Peer_ASN': peer_asn,
                'Prefix': prefix,
                'AS_Path': as_path,
                'Origin': origin,
                'Next_Hop': next_hop,
                'MED': med,
                'Local_Pref': local_pref,
                'Communities': communities,
                'Label': 'normal'
            })
        
        # Withdrawals
        for wd in update.withdrawn_routes:
            prefix = wd.prefix if hasattr(wd, 'prefix') else str(wd)
            rows.append({
                'Time': time_str,
                'Entry_Type': 'W',
                'Peer_IP': peer_ip,
                'Peer_ASN': peer_asn,
                'Prefix': prefix,
                'AS_Path': '',
                'Origin': '',
                'Next_Hop': '',
                'MED': '',
                'Local_Pref': '',
                'Communities': '',
                'Label': 'normal'
            })
    
    if rows:
        import pandas as pd
        df = pd.DataFrame(rows)
        df.to_csv(csv_file, index=False)
        print(f'✅ CSV saved to {csv_file} ({len(rows)} entries)')
        return df
    else:
        print('❌ No BGP UPDATE packets found')
        return None

df = extract_bgp_to_csv(OUTPUT_PCAP, OUTPUT_CSV)
if df is not None:
    print(f'\nCSV Summary:')
    print(f'  Total entries: {len(df)}')
    print(f'  Announcements: {len(df[df["Entry_Type"]=="A"])}')
    print(f'  Withdrawals: {len(df[df["Entry_Type"]=="W"])}')

In [ ]:
# ======================================
# PART 8: Validate Correlations
# Run this to check if correlations are improved

import pandas as pd
import numpy as np

# Use the feature extraction code to check correlations
# Import from feature_exctration_v2 or use inline

print('='*60)
print('CORRELATION VALIDATION')
print('='*60)
print()
print('To validate correlations, run feature_exctration_v2.ipynb on')
print(f'the generated CSV file: {OUTPUT_CSV}')
print()
print('Expected improvements:')
print('-' * 60)
for (f1, f2), target in REAL_CORRELATIONS.items():
    print(f'  {f1:25} ↔ {f2:25}: {target:.3f}')
print('='*60)